In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install pandas-datareader

In [7]:
import pandas_datareader as web
import pandas as pd

In [10]:
start = "2000-01-01"
end = "2026-03-06"

wti = web.DataReader("DCOILWTICO", "fred", start, end)

In [19]:
wti = wti.reset_index()
wti['day'] = wti['DATE'].dt.day
wti['month'] = wti['DATE'].dt.month
wti['year'] = wti['DATE'].dt.year

In [22]:
wti = wti.dropna(axis=0)

In [29]:
wti = wti.rename(columns={"DATE":"datetime", "DCOILWTICO":"oil_price"})
wti["datetime"] = pd.to_datetime(wti["datetime"])

In [30]:
wti

,datetime,oil_price,day,month,year
1,2000-01-04,25.56,4,1,2000
2,2000-01-05,24.65,5,1,2000
3,2000-01-06,24.79,6,1,2000
4,2000-01-07,24.79,7,1,2000
5,2000-01-10,24.71,10,1,2000
...,...,...,...,...,...
6821,2026-02-24,65.62,24,2,2026
6822,2026-02-25,65.30,25,2,2026
6823,2026-02-26,65.10,26,2,2026
6824,2026-02-27,66.96,27,2,2026


In [11]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 18.9 MB/s eta 0:00:00


In [13]:
from pymongo import MongoClient
import pandas as pd

# Connect to MongoDB
MONGO_URI = input("Enter mongo uri: ")
client = MongoClient(MONGO_URI)  # or Atlas URI
db = client["test"]
collection = db["news"]

In [38]:
# Load collection into DataFrame
vessels = pd.DataFrame(list(collection.find()))

# Optional: drop the _id column
vessels = vessels.drop(columns=["_id"])

In [17]:
# Preview
print(vessels.head())
print(vessels.shape)  # (rows, columns)

# Save to CSV
vessels.to_csv("output.csv", index=False)
print("✅ Saved to output.csv")

    eventDate  year  month country countryCode            region  \
0  2000-01-17  2000      1     UAE         ARE     Red Sea Route   
1  2000-02-04  2000      2    Iran         IRN  Strait of Hormuz   
2  2000-01-06  2000      1    Iraq         IRQ  Strait of Hormuz   
3  2000-02-10  2000      2   Qatar         QAT      Persian Gulf   
4  2000-02-09  2000      2    Oman         OMN       Arabian Sea   

                eventType  severityScore  riskScore  predictedOilMovePct48h  \
0      Shipping Lane Risk           0.82       82.0                    3.75   
1      Shipping Lane Risk           0.67       67.0                   -1.96   
2     Pipeline Disruption           0.55       55.0                    1.23   
3  Geopolitical Statement           0.34       34.0                   -0.84   
4  Geopolitical Statement           0.21       21.0                   -2.16   

       asset      source                                           headline  
0  Crude Oil  Al Jazeera  Shipping Lan

In [39]:
vessels["eventDate"] = pd.to_datetime(vessels["eventDate"])

In [40]:
for i in range(1, 8):
    vessels[f'oil_{i}d'] = vessels['eventDate'].apply(
        lambda x: wti.loc[wti['datetime'] == x + pd.Timedelta(days=i), 'oil_price'].values[0]
        if (x + pd.Timedelta(days=i)) in wti['datetime'].values else None
    )

In [42]:
vessels["day"] = vessels["eventDate"].dt.day

In [46]:
vessels = vessels.sort_values(by="eventDate")

In [47]:
vessels

,eventDate,year,month,country,countryCode,region,eventType,severityScore,riskScore,predictedOilMovePct48h,...,source,headline,oil_1d,oil_2d,oil_3d,oil_4d,oil_5d,oil_6d,oil_7d,day
24,2000-01-02,2000,1,Saudi Arabia,SAU,Arabian Sea,Shipping Lane Risk,0.66,66.0,-2.12,...,Bloomberg,Shipping Lane Risk reported near Arabian Sea a...,NaN,25.56,24.65,24.79,24.79,NaN,NaN,2
6,2000-01-04,2000,1,Qatar,QAT,Persian Gulf,OPEC Production Decision,0.42,42.0,2.20,...,Bloomberg,OPEC Production Decision reported near Persian...,24.65,24.79,24.79,NaN,NaN,24.71,25.69,4
2,2000-01-06,2000,1,Iraq,IRQ,Strait of Hormuz,Pipeline Disruption,0.55,55.0,1.23,...,AP,Pipeline Disruption reported near Strait of Ho...,24.79,NaN,NaN,24.71,25.69,26.30,26.63,6
17,2000-01-06,2000,1,Iraq,IRQ,Strait of Hormuz,Sanctions Announcement,0.30,30.0,0.67,...,Bloomberg,Sanctions Announcement reported near Strait of...,24.79,NaN,NaN,24.71,25.69,26.30,26.63,6
16,2000-01-10,2000,1,UAE,ARE,Red Sea Route,Shipping Lane Risk,0.38,38.0,-1.90,...,Financial Times,Shipping Lane Risk reported near Red Sea Route...,25.69,26.30,26.63,28.01,NaN,NaN,NaN,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3233,2026-12-25,2026,12,Saudi Arabia,SAU,Red Sea Route,Geopolitical Statement,0.32,32.0,2.97,...,Al Jazeera,Geopolitical Statement reported near Red Sea R...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25
3226,2026-12-26,2026,12,Iran,IRN,Arabian Sea,Drone Attack,0.46,46.0,1.95,...,Reuters,Drone Attack reported near Arabian Sea affecti...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26
3232,2026-12-28,2026,12,Saudi Arabia,SAU,Red Sea Route,Port Disruption,0.21,21.0,1.13,...,Bloomberg,Port Disruption reported near Red Sea Route af...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28
3235,2026-12-28,2026,12,Oman,OMN,Persian Gulf,Pipeline Disruption,0.67,67.0,0.74,...,Financial Times,Pipeline Disruption reported near Persian Gulf...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28


In [99]:
dep_vars = ["oil_1d", "oil_2d", "oil_3d", "oil_4d", "oil_5d", "oil_6d", "oil_7d"]
ind_vars = vessels.drop(columns=dep_vars)

In [52]:
vessels.set_index("eventDate", inplace=True)

In [59]:
vessels

,year,month,country,countryCode,region,eventType,severityScore,riskScore,predictedOilMovePct48h,asset,source,headline,oil_1d,oil_2d,oil_3d,oil_4d,oil_5d,oil_6d,oil_7d,day
eventDate,,,,,,,,,,,,,,,,,,,,
2000-01-02,2000,1,Saudi Arabia,SAU,Arabian Sea,Shipping Lane Risk,0.66,66.0,-2.12,Crude Oil,Bloomberg,Shipping Lane Risk reported near Arabian Sea a...,NaN,25.56,24.65,24.79,24.79,NaN,NaN,2
2000-01-04,2000,1,Qatar,QAT,Persian Gulf,OPEC Production Decision,0.42,42.0,2.20,Crude Oil,Bloomberg,OPEC Production Decision reported near Persian...,24.65,24.79,24.79,NaN,NaN,24.71,25.69,4
2000-01-06,2000,1,Iraq,IRQ,Strait of Hormuz,Pipeline Disruption,0.55,55.0,1.23,Crude Oil,AP,Pipeline Disruption reported near Strait of Ho...,24.79,NaN,NaN,24.71,25.69,26.30,26.63,6
2000-01-06,2000,1,Iraq,IRQ,Strait of Hormuz,Sanctions Announcement,0.30,30.0,0.67,Crude Oil,Bloomberg,Sanctions Announcement reported near Strait of...,24.79,NaN,NaN,24.71,25.69,26.30,26.63,6
2000-01-10,2000,1,UAE,ARE,Red Sea Route,Shipping Lane Risk,0.38,38.0,-1.90,Crude Oil,Financial Times,Shipping Lane Risk reported near Red Sea Route...,25.69,26.30,26.63,28.01,NaN,NaN,NaN,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-12-25,2026,12,Saudi Arabia,SAU,Red Sea Route,Geopolitical Statement,0.32,32.0,2.97,Crude Oil,Al Jazeera,Geopolitical Statement reported near Red Sea R...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25
2026-12-26,2026,12,Iran,IRN,Arabian Sea,Drone Attack,0.46,46.0,1.95,Crude Oil,Reuters,Drone Attack reported near Arabian Sea affecti...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26
2026-12-28,2026,12,Saudi Arabia,SAU,Red Sea Route,Port Disruption,0.21,21.0,1.13,Crude Oil,Bloomberg,Port Disruption reported near Red Sea Route af...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28


In [67]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [81]:
other_categorical_cols = ind_vars.select_dtypes(include=["object", "category"]).drop(columns=["headline"]).columns
numeric_cols = ind_vars.select_dtypes(include="number").columns

In [93]:
sentences = ind_vars["headline"].tolist()
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_embeddings = model.encode(sentences)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), other_categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ],
    remainder='drop'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [95]:
X_other = preprocessor.fit_transform(ind_vars.drop(columns=["headline"]))

In [96]:
exog = np.hstack([X_other, sentence_embeddings])

In [100]:
models = {}
for dep_var in dep_vars:
  y = vessels[dep_var]
  model = SARIMAX(y, exog=exog, order=(1,1,1))
  model_fit = model.fit(disp=False)

  models[dep_var] = model_fit

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/di

In [101]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [102]:
import pickle

pln = {
    "models" : models,
    "preprocessor" : preprocessor,
    "embed_model" : sentence_embeddings
}

path = "/content/drive/MyDrive/oil_news_models.pkl"

pickle.dump(pln, open("/content/drive/MyDrive/oil_news_pipeline.pkl","wb"))

In [104]:
path2 = path = "/content/drive/MyDrive/oil_news_pipeline.pkl"

path = "/content/drive/MyDrive/oil_news_pipeline.pkl"

pipeline = pickle.load(open(path, "rb"))
models = pipeline["models"]
preprocessor = pipeline["preprocessor"]

In [122]:
sentence_data = vessels["headline"]
data = vessels[ind_vars.drop(columns=["headline"]).columns]

In [123]:
embeddings = pipeline["embed_model"]
X = preprocessor.transform(data)

In [128]:
predictions_1d = []
predictions_2d = []
predictions_3d = []
predictions_4d = []
predictions_5d = []
predictions_6d = []
predictions_7d = []

for i in range(0, len(exog)):
  predictions_1d.append(models["oil_1d"].forecast(steps=1, exog=exog[0]))
  predictions_2d.append(models["oil_2d"].forecast(steps=1, exog=exog[0]))
  predictions_3d.append(models["oil_3d"].forecast(steps=1, exog=exog[0]))
  predictions_4d.append(models["oil_4d"].forecast(steps=1, exog=exog[0]))
  predictions_5d.append(models["oil_5d"].forecast(steps=1, exog=exog[0]))
  predictions_6d.append(models["oil_6d"].forecast(steps=1, exog=exog[0]))
  predictions_7d.append(models["oil_7d"].forecast(steps=1, exog=exog[0]))

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/li

In [129]:
predictions_df = pd.DataFrame({
    'oil_1d_pred': [p.iloc[0] for p in predictions_1d],
    'oil_2d_pred': [p.iloc[0] for p in predictions_2d],
    'oil_3d_pred': [p.iloc[0] for p in predictions_3d],
    'oil_4d_pred': [p.iloc[0] for p in predictions_4d],
    'oil_5d_pred': [p.iloc[0] for p in predictions_5d],
    'oil_6d_pred': [p.iloc[0] for p in predictions_6d],
    'oil_7d_pred': [p.iloc[0] for p in predictions_7d]
})

In [131]:
predictions_df.to_csv("/content/drive/MyDrive/oil_price_predictions.csv", index=False)
print("✅ Predictions saved to oil_price_predictions.csv")
predictions_df.head()

✅ Predictions saved to oil_price_predictions.csv


,oil_1d_pred,oil_2d_pred,oil_3d_pred,oil_4d_pred,oil_5d_pred,oil_6d_pred,oil_7d_pred
0,-1529.564988,-303.470379,36.293763,61.976893,-518.491445,-4077.010279,3353.400816
1,-1529.564988,-303.470379,36.293763,61.976893,-518.491445,-4077.010279,3353.400816
2,-1529.564988,-303.470379,36.293763,61.976893,-518.491445,-4077.010279,3353.400816
3,-1529.564988,-303.470379,36.293763,61.976893,-518.491445,-4077.010279,3353.400816
4,-1529.564988,-303.470379,36.293763,61.976893,-518.491445,-4077.010279,3353.400816


In [132]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [134]:
import os
os.path.exists("/content/drive/MyDrive/ais_combined.csv")

False